<h1 style="text-align: center;">FinPay Marketing Analytics (Jan - March 2026)</h1>

<p style="text-align: center; font-size: 18px;">
Optimizing Customer Acquisition and Conversion
</p>

## Overview

This notebook analyzes FinPay Digital’s marketing performance using SQL queries, with a focus on campaign spend, user engagement, and conversion outcomes.

The goal is to evaluate how effectively marketing efforts drive customer acquisition and identify opportunities for improving campaign efficiency and return on investment.

## Business Context

FinPay Digital is a digital payments platform that provides services such as digital wallets, peer-to-peer transfers, merchant payment solutions, bill payments, and POS transactions.

The platform operates within the digital payments industry, where growth depends heavily on acquiring and retaining active users across multiple digital channels.

As a result, effective marketing and customer acquisition strategies are critical to platform expansion and user adoption.

## Problem Focus

- Inefficient allocation of marketing budget  
- Limited insight into campaign effectiveness  
- Difficulty identifying high-performing channels  
- Inconsistent conversion performance  

## Analytical Objectives

This analysis aims to evaluate FinPay Digital’s marketing performance using SQL-based analysis to generate actionable insights for improving customer acquisition and conversion efficiency.

Specifically, the objectives are to:

- Evaluate campaign performance across different marketing channels to identify the most effective acquisition sources  
- Identify high-performing campaigns and creatives based on engagement and conversion metrics  
- Measure key performance indicators (KPIs) such as Click-Through Rate (CTR), Customer Acquisition Cost (CAC), Return on Investment (ROI), and Conversion Rate  
- Analyze campaign-level engagement patterns to understand performance differences across channels  
- Generate insights to support optimization of future marketing strategies

## Dataset Overview

The FinPay marketing dataset is a simulated representation of a digital advertising system within a fintech environment. It is designed to reflect how marketing campaigns, user interactions, and performance metrics are structured and tracked in real-world acquisition systems.

The dataset was synthetically generated using domain knowledge, AI-assisted brainstorming, and manual data modeling to simulate realistic marketing behavior across multiple channels and user engagement touchpoints.

The dataset consists of several relational tables, including:

- **Campaigns**: Contains details of marketing campaigns such as objectives, timelines, and targeting structure  
- **Ads**: Stores information about individual advertisements and creative variations used in campaigns  
- **Users**: Contains user-level information representing individuals exposed to marketing activities  
- **Channels**: Defines the marketing platforms or acquisition sources used for campaigns  
- **Campaign Spend**: Tracks daily expenditure associated with campaigns  
- **Impressions**: Records the number of times ads were displayed to users  
- **Clicks**: Captures user interactions in the form of ad clicks  
- **Ad Engagement**: Tracks deeper engagement actions beyond clicks  
- **Conversions**: Records final user actions such as sign-ups, purchases, or other conversion events  

These tables provide a structured foundation for analyzing marketing performance across the customer journey from exposure to conversion.

## Entity Relationship (ER) Diagram

The ER diagram below illustrates how the core entities in the FinPay marketing database are connected, showing the relationships between users, campaigns, ads, and marketing activity tables used for analysis.

![ER Diagram](finpay_er_diagram.png)

## Database Setup

The dataset for this analysis resides in a SQL Server database and is queried directly within the notebook environment. A connection is established to enable the execution of SQL queries for data extraction and analysis.

This setup allows data retrieval and analytical processes to be carried out seamlessly within a single workspace.

## Connecting to SQL Server database

In [1]:
# 1️⃣ Import required libraries for data analysis, database connection, and system control
import pandas as pd
from sqlalchemy import create_engine
import urllib
import warnings
import logging

# 2️⃣ Suppress warnings and SQLAlchemy logs for cleaner notebook output
warnings.filterwarnings("ignore")
logging.getLogger("sqlalchemy.engine").setLevel(logging.ERROR)

# 3️⃣ Define SQL Server connection string
# Includes driver, server instance, database name, and authentication method
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=finpay_marketing_data;"
    "Trusted_Connection=yes;"
)

# 4️⃣ Encode connection string for SQLAlchemy compatibility
# Ensures special characters are safely formatted
params = urllib.parse.quote_plus(connection_string)

# 5️⃣ Create database engine
# Acts as the bridge between Python (pandas) and SQL Server
engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=%s" % params
)

# 6️⃣ Test database connection
# Runs a simple query to confirm successful connection
with engine.connect() as conn:
    df = pd.read_sql_query("SELECT 1 AS test", conn)

df

,test
0,1


## Analyzing the FinPay Marketing Dataset

With the FinPay marketing database connected, we can begin to analyse the data to gain insights into user activity, campaign performance, and ad engagement across digital channels. This helps evaluate how effectively these marketing efforts drive customer acquisition.

This analysis focuses on assessing campaign performance, identifying high-converting ads and audience segments, and measuring key marketing KPIs such as Click-Through Rate (CTR), Conversion Rate, Customer Acquisition Cost (CAC), and Return On Investment (ROI). The goal is to highlight performance gaps and support more efficient allocation of marketing resources.

### Channel Acquisition Effectiveness:

This analysis evaluates the effectiveness of FinPay Digital’s marketing channels in driving customer acquisition across users and merchants.

1. Which acquisition channels achieve the highest interaction rates based on ad engagement metrics?

In [4]:
pd.read_sql("""
WITH engagement AS (                          -- Aggregate total user interactions (clicks, likes, etc.) per campaign
    SELECT 
        campaign_id, 
        SUM(interaction_count) AS total_engagement
    FROM 
        fact_ad_engagements
    GROUP BY 
        campaign_id
),

impression AS (                              -- Count total ad impressions per campaign
    SELECT 
        campaign_id, 
        COUNT(impression_id) AS total_impression
    FROM 
        fact_impression
    GROUP BY 
        campaign_id
),

channel_info AS (                            -- Map each campaign to its corresponding marketing channel
    SELECT
        camp.campaign_id, 
        ch.channel_id, 
        ch.channel_name
    FROM 
        dim_channel AS ch
    JOIN 
        dim_campaign AS camp
    ON 
        ch.channel_id = camp.channel_id
)

SELECT                                     -- Calculate interaction rate across marketing channels by comparing total engagement to total impressions
    cha.channel_id,
    cha.channel_name,
    ROUND(SUM(eng.total_engagement) * 100.0 
            / SUM(imp.total_impression), 
    2) AS interaction_rate
FROM 
    channel_info AS cha
LEFT JOIN 
    engagement AS eng
ON 
    cha.campaign_id = eng.campaign_id
LEFT JOIN 
    impression AS imp
ON 
    cha.campaign_id = imp.campaign_id
GROUP BY
    cha.channel_id,
    cha.channel_name
ORDER BY 
    interaction_rate DESC;
""", engine)

,channel_id,channel_name,interaction_rate
0,CH-004,Referral Program,75.98
1,CH-001,Meta Ads,75.14
2,CH-003,Tiktok Ads,75.13
3,CH-002,Google Ads,73.57
4,CH-005,Email Marketing,62.88


*Referral Program achieved the highest interaction rate (75.98%), indicating strong user engagement driven by trust-based or incentive-led acquisition, while Email Marketing recorded the lowest interaction rate (62.88%), indicating weaker engagement efficiency, possibly due to lower responsiveness or message fatigue compared to social and referral channels.*

2. Which marketing channels deliver the highest conversion rate from impressions to user sign-ups?

In [5]:
pd.read_sql("""
WITH signup_conversion AS (              -- Count total signup conversions per campaign.
    SELECT
        campaign_id, 
        COUNT(conversion_id) AS total_signup_conversion
    FROM
        fact_conversion
    WHERE
        conversion_type = 'Signup'
    GROUP BY 
        campaign_id
),

impression_count AS (                    -- Count total impressions per campaign.
    SELECT 
        campaign_id,
        COUNT(impression_id) AS total_impression
    FROM
        fact_impression
    GROUP BY 
        campaign_id
),

channel_inf AS (                        -- Retrieve campaign and associated marketing channel information.
    SELECT
        camp.campaign_id, 
        ch.channel_id, 
        ch.channel_name
    FROM 
        dim_channel AS ch
    LEFT JOIN 
        dim_campaign AS camp
    ON 
        ch.channel_id = camp.channel_id
)

SELECT                                 -- Retrieve channels with high conversion rate by comparing total signup conversions against total impressions.                                          
    cha.channel_id,
    cha.channel_name,
    SUM(total_impression) AS total_impression,
    SUM(total_signup_conversion) AS total_signup_conversion,
    ROUND(SUM(con.total_signup_conversion) * 100.0 
            / NULLIF(SUM(imp.total_impression), 0), 
    2) AS conversion_rate
FROM 
    channel_inf AS cha
LEFT JOIN 
    signup_conversion AS con
ON 
    cha.campaign_id = con.campaign_id
LEFT JOIN 
    impression_count AS imp
ON
    imp.campaign_id = cha.campaign_id
GROUP BY
    cha.channel_id,
    cha.channel_name
ORDER BY
    conversion_rate DESC;
""", engine)

,channel_id,channel_name,total_impression,total_signup_conversion,conversion_rate
0,CH-001,Meta Ads,47217,81,0.17
1,CH-004,Referral Program,34416,41,0.12
2,CH-005,Email Marketing,31721,11,0.03
3,CH-002,Google Ads,55777,16,0.03
4,CH-003,Tiktok Ads,30869,6,0.02


*Meta Ads achieved the highest signup conversion rate at 0.17%, while Google Ads recorded the highest impressions but a relatively low signup conversion rate of 0.03%, suggesting the channel may have been more effective in driving other conversion types beyond user sign-ups.*

### Campaign Return on Investment:

This analysis evaluates the return on marketing spend by identifying which campaigns generate the highest value relative to their cost.

1. Which campaigns generate the highest ROI relative to marketing spend?

KPI Definition (ROI) - ROI is calculated using total conversion value as a proxy for revenue, excluding signup conversions.

In [7]:
pd.read_sql("""
With kpi_cte AS (             -- Compute total conversion value per campaign as revenue proxy and prepare data for ROI analysis against campaign spend.
        SELECT 
            cam.campaign_id,
            campaign_name,
            budget AS campaign_spend,
            SUM(conversion_value) AS total_conversion_value
        FROM 
            dim_campaign AS cam
        LEFT JOIN 
            fact_conversion AS con
        ON 
            cam.campaign_id = con.campaign_id 
               AND conversion_type IN ('Bills', 'Deposit', 'Merchant', 'Transfer', 'Airtime')
        GROUP BY 
            cam.campaign_id, campaign_name, budget
)

SELECT                      -- Calculate ROI per campaign to evaluate marketing efficiency based on revenue generated relative to campaign spend.
     campaign_id,
     campaign_name,
     campaign_spend,
     total_conversion_value,
     ((total_conversion_value - campaign_spend) * 100) / campaign_spend AS ROI
FROM 
    kpi_cte
ORDER BY 
    total_conversion_value DESC;
""", engine)

,campaign_id,campaign_name,campaign_spend,total_conversion_value,ROI
0,CMP-002,Cashback Boost,4200000,3977000,-5
1,CMP-005,Card Activation,4800000,2195000,-54
2,CMP-004,Win-Back Users,2500000,1802000,-27
3,CMP-001,New Year Start Smart,5000000,1269500,-74
4,CMP-003,Refer & Earn,3000000,453000,-84
5,CMP-006,Trust & Security,3500000,322000,-90


*The ROI analysis shows that all campaigns generated negative estimated ROI when evaluated using total conversion value as a proxy for revenue. This indicates that campaign spend currently exceeds the immediate transactional value generated from acquired users. Among all campaigns, Cashback Boost (CMP-002) performed best with the least negative ROI (-5%), while Trust & Security (CMP-006) recorded the lowest performance (-90%), highlighting significant variation in campaign efficiency across acquisition strategies.*

2. How do key performance metrics (CTR and conversion rate) vary across campaigns?

In [8]:
pd.read_sql("""
WITH impression_cte AS (        -- Total impressions per campaign
    SELECT 
        campaign_id,
        COUNT(impression_id) AS total_impression
    FROM 
        fact_impression
    GROUP BY 
        campaign_id
),

click_cte AS (                   -- Total clicks per campaign
    SELECT 
        campaign_id,
        COUNT(click_id) AS total_click
    FROM 
        fact_click
    GROUP BY 
        campaign_id
),

conversion_cte AS (              -- Total conversions per campaign
    SELECT 
        campaign_id,
        COUNT(conversion_id) AS total_conversion
    FROM 
        fact_conversion
    GROUP BY 
        campaign_id
)

SELECT                           -- Calculate CTR and CVR across all campaign
    cam.campaign_id,
    cam.campaign_name,
    ROUND((cli.total_click * 100.0) 
            / imp.total_impression,  2) AS click_through_rate,
    ROUND((con.total_conversion * 100.0) 
            / cli.total_click, 2) AS conversion_rate
FROM 
    dim_campaign AS cam
LEFT JOIN 
    impression_cte AS imp
ON 
    cam.campaign_id = imp.campaign_id
LEFT JOIN 
    click_cte AS cli
ON 
    cam.campaign_id = cli.campaign_id
LEFT JOIN 
    conversion_cte AS con
ON 
    cam.campaign_id = con.campaign_id
ORDER BY 
    cam.campaign_id
""", engine)

,campaign_id,campaign_name,click_through_rate,conversion_rate
0,CMP-001,New Year Start Smart,2.98,7.68
1,CMP-002,Cashback Boost,2.99,7.63
2,CMP-003,Refer & Earn,2.92,5.57
3,CMP-004,Win-Back Users,3.03,6.87
4,CMP-005,Card Activation,3.01,7.22
5,CMP-006,Trust & Security,3.20,6.28


*CTR is relatively stable across campaigns (~3%), suggesting that ad creatives have similar effectiveness in driving initial engagement. However, variation in conversion rates (5.57% – 7.68%) indicates that post-click experience and offer relevance are the primary drivers of campaign performance differences. This suggests that optimization efforts should focus less on ad visibility and more on improving landing flow and targeting quality for lower-performing campaigns such as Refer & Earn (CMP-003).*

3. Which campaigns generate the highest number of new users relative to their reach and marketing spend?

In [9]:
pd.read_sql("""
WITH conversion_det AS (                      -- Count total signup conversions per campaign
    SELECT 
        campaign_id,
        COUNT(conversion_id) AS total_conversion
    FROM 
        fact_conversion
    WHERE
        conversion_type = 'Signup'
    GROUP BY
        campaign_id
),

impression_info AS (                          -- Count total impressions per campaign to measure campaign reach
    SELECT 
        campaign_id,
        COUNT(impression_id) AS total_impression
    FROM 
        fact_impression
    GROUP BY
        campaign_id
)

SELECT                                         -- Evaluate campaign efficiency based on signup conversions relative to impressions and budget
    camp.campaign_id,
    campaign_name,
    budget,
    COALESCE(total_conversion, 0) AS total_conversion,
    total_impression
FROM
    dim_campaign AS camp
LEFT JOIN 
    conversion_det AS con
ON
    camp.campaign_id = con.campaign_id
LEFT JOIN
    impression_info AS imp
ON
    camp.campaign_id = imp.campaign_id
ORDER BY
    total_conversion DESC
""", engine)

,campaign_id,campaign_name,budget,total_conversion,total_impression
0,CMP-001,New Year Start Smart,5000000,81,47217
1,CMP-003,Refer & Earn,3000000,41,34416
2,CMP-006,Trust & Security,3500000,16,15456
3,CMP-004,Win-Back Users,2500000,11,31721
4,CMP-005,Card Activation,4800000,6,30869
5,CMP-002,Cashback Boost,4200000,0,40321


*The campaign analysis shows variation in performance across campaigns when evaluated using signup conversions, with New Year Start Smart (CMP-001) recording the highest number of signups, making it the most effective campaign for this conversion type. Refer & Earn (CMP-003) also shows relatively strong signup activity compared to other campaigns. However, when considering other conversion types beyond signups, Cashback Boost (CMP-002) stands out as the top-performing campaign in terms of conversion value, indicating strong performance in non-signup user actions despite weak signup output.*

### Customer Journey Conversion Efficiency

This analysis evaluates how effectively users move through the marketing funnel from ad impressions to clicks and final conversions. It identifies the stages where users drop off most frequently and assesses how conversion performance varies across different channels and campaigns to measure the overall efficiency of the customer journey.

1. What are the conversion rates across the marketing funnel (impressions → clicks → conversions)?

In [12]:
pd.read_sql("""
WITH tot_impression AS (                    -- Count total impressions per campaign (top-of-funnel reach)
    SELECT                             
        campaign_id,
        COUNT(impression_id) AS total_impression
    FROM 
        fact_impression
    GROUP BY 
        campaign_id
),

tot_click AS (                              -- Count total clicks per campaign (user engagement stage)
    SELECT
        campaign_id,
        COUNT(click_id) AS total_click
    FROM 
        fact_click
    GROUP BY 
        campaign_id
),

tot_conversion AS (                          -- Count total conversions per campaign (final action stage)
    SELECT 
        campaign_id,
        COUNT(conversion_id) AS total_conversion
    FROM 
        fact_conversion
    GROUP BY 
        campaign_id
),

channel_info AS (                           -- Map campaigns to their respective marketing channels
    SELECT
        camp.campaign_id, 
        ch.channel_id, 
        ch.channel_name
    FROM 
        dim_campaign AS camp
    JOIN 
        dim_channel AS ch
    ON 
        ch.channel_id = camp.channel_id
)

SELECT                                        -- Analyze end-to-end funnel performance (impressions → clicks → conversions) across marketing channels
    ci.channel_id,
    ci.channel_name,
    SUM(imp.total_impression) AS total_impressions,
    SUM(cli.total_click) AS total_clicks,
    SUM(con.total_conversion) AS total_conversions,
    ROUND(SUM(cli.total_click) * 100.0 
            / NULLIF(SUM(imp.total_impression), 0), 
    2) AS ctr_percentage, 
    
    ROUND(SUM(con.total_conversion) * 100.0 
            / NULLIF(SUM(cli.total_click), 0), 
    2) AS conversion_rate_percentage,
    
    ROUND(SUM(con.total_conversion) * 100.0 
            / NULLIF(SUM(imp.total_impression), 0), 
    2) AS overall_funnel_conversion_percentage
FROM 
    channel_info ci
LEFT JOIN 
    tot_impression imp
ON 
    ci.campaign_id = imp.campaign_id
LEFT JOIN 
    tot_click cli
ON 
    ci.campaign_id = cli.campaign_id
LEFT JOIN 
    tot_conversion con
ON 
    ci.campaign_id = con.campaign_id
GROUP BY
    ci.channel_id,
    ci.channel_name
ORDER BY
    overall_funnel_conversion_percentage DESC;
""", engine)

,channel_id,channel_name,total_impressions,total_clicks,total_conversions,ctr_percentage,conversion_rate_percentage,overall_funnel_conversion_percentage
0,CH-001,Meta Ads,47217,1406,108,2.98,7.68,0.23
1,CH-002,Google Ads,55777,1699,123,3.05,7.24,0.22
2,CH-003,Tiktok Ads,30869,928,67,3.01,7.22,0.22
3,CH-005,Email Marketing,31721,961,66,3.03,6.87,0.21
4,CH-004,Referral Program,34416,1006,56,2.92,5.57,0.16


*All channels have very similar CTRs (~3%), which shows that users respond to ads fairly consistently across platforms at the impression-to-click stage. However, there is a clear drop in performance after clicks, where Meta Ads and Google Ads convert slightly better than other channels, while Referral Program performs the weakest across both click-to-conversion and overall funnel efficiency. Overall, the results indicate that the main differences in performance are not in attracting clicks, but in converting those clicks into actual conversions.*

2. At which stage of the funnel do users drop off most frequently?

In [9]:
pd.read_sql("""
WITH tot_impression AS (                 -- Count total impressions per campaign (top-of-funnel exposure)
    SELECT
        campaign_id,
        COUNT(impression_id) AS total_impressions
    FROM 
        fact_impression
    GROUP BY campaign_id
),

tot_click AS (                           -- Count total clicks per campaign (user engagement stage)
    SELECT
        campaign_id,
        COUNT(click_id) AS total_clicks
    FROM fact_click
    GROUP BY campaign_id
),

tot_conversion AS (                      -- Count total conversions per campaign (final action stage)
    SELECT 
        campaign_id,
        COUNT(conversion_id) AS total_conversions
    FROM fact_conversion
    GROUP BY campaign_id
),

channel_info AS (                        -- Map each campaign to its corresponding marketing channel
    SELECT
        camp.campaign_id, 
        ch.channel_id, 
        ch.channel_name
    FROM 
        dim_campaign AS camp
    JOIN 
        dim_channel AS ch
    ON 
        ch.channel_id = camp.channel_id
),

funnel AS (                              -- Combine impression, click, and conversion data to build full funnel metrics per channel
    SELECT
        ci.channel_id,
        ci.channel_name,
        imp.total_impressions,
        cli.total_clicks,
        con.total_conversions
    FROM 
        channel_info AS ci
    LEFT JOIN 
        tot_impression imp
    ON 
        ci.campaign_id = imp.campaign_id
    LEFT JOIN 
        tot_click AS cli
    ON 
        ci.campaign_id = cli.campaign_id
    LEFT JOIN 
        tot_conversion AS con
    ON 
        ci.campaign_id = con.campaign_id
)

SELECT                                    -- Analyze funnel drop-off rates across marketing channels (impressions → clicks → conversions)
    channel_id,
    channel_name,
    SUM(total_impressions) AS impressions,
    SUM(total_clicks) AS clicks,
    SUM(total_conversions) AS conversions,
    ROUND(
        (SUM(total_impressions) - SUM(total_clicks)) * 100.0 
            / NULLIF(SUM(total_impressions), 0), 
    2) AS dropoff_impression_to_click_pct,
    
    ROUND(
        (SUM(total_clicks) - SUM(total_conversions)) * 100.0 
            / NULLIF(SUM(total_clicks), 0), 
    2) AS dropoff_click_to_conversion_pct,
    
    ROUND(
        (SUM(total_impressions) - SUM(total_conversions)) * 100.0 
            / NULLIF(SUM(total_impressions), 0), 
    2) AS overall_dropoff_pct
FROM 
    funnel
GROUP BY 
    channel_id, channel_name
ORDER BY 
    overall_dropoff_pct DESC;
""", engine)

,channel_id,channel_name,impressions,clicks,conversions,dropoff_impression_to_click_pct,dropoff_click_to_conversion_pct,overall_dropoff_pct
0,CH-004,Referral Program,34416,1006,56,97.08,94.43,99.84
1,CH-005,Email Marketing,31721,961,66,96.97,93.13,99.79
2,CH-002,Google Ads,55777,1699,123,96.95,92.76,99.78
3,CH-003,Tiktok Ads,30869,928,67,96.99,92.78,99.78
4,CH-001,Meta Ads,47217,1406,108,97.02,92.32,99.77


*All channels show a very similar funnel pattern, with the largest drop-off occurring at the impression-to-click stage (97%), which is expected in digital advertising funnels where only a small percentage of users typically engage after seeing an ad. The next biggest drop happens at the click-to-conversion stage (92–94%), but it is less severe compared to the initial drop. Overall, channel performance differences are minimal, indicating that the funnel issue is not channel-specific but a general top-of-funnel engagement problem across all platforms.*

3. How does funnel performance vary across channels and campaigns?

In [10]:
pd.read_sql("""
WITH tot_impression AS (         -- Count total impressions per campaign (top-of-funnel reach)
    SELECT
        campaign_id,
        COUNT(impression_id) AS total_impressions
    FROM 
        fact_impression
    GROUP 
        BY campaign_id
),

tot_click AS (                   -- Count total clicks per campaign (user engagement stage)
    SELECT
        campaign_id,
        COUNT(click_id) AS total_clicks
    FROM 
        fact_click
    GROUP BY 
        campaign_id
),

tot_conversion AS (              -- Count total conversions per campaign (final action stage)
    SELECT 
        campaign_id,
        COUNT(conversion_id) AS total_conversions
    FROM 
        fact_conversion
    GROUP BY 
        campaign_id
),

campaign_info AS (              -- Map each campaign to its marketing channel for combined campaign-channel analysis
    SELECT
        camp.campaign_id,
        camp.campaign_name,
        ch.channel_id,
        ch.channel_name
    FROM 
        dim_campaign camp
    JOIN 
        dim_channel ch
    ON 
        camp.channel_id = ch.channel_id
),

funnel AS (                    -- Combine impressions, clicks, and conversions into a unified funnel dataset
    SELECT
        ci.channel_id,
        ci.channel_name,
        ci.campaign_id,
        ci.campaign_name,
        imp.total_impressions,
        cli.total_clicks,
        con.total_conversions
    FROM 
        campaign_info ci
    LEFT JOIN 
        tot_impression imp
    ON 
        ci.campaign_id = imp.campaign_id
    LEFT JOIN 
        tot_click cli
    ON 
        ci.campaign_id = cli.campaign_id
    LEFT JOIN 
        tot_conversion con
    ON 
        ci.campaign_id = con.campaign_id
)

SELECT                        -- Analyse funnel performance at campaign and channel level using CTR, conversion rate, and overall conversion efficiency
    channel_id,
    channel_name,
    campaign_id,
    campaign_name,
    total_impressions,
    total_clicks,
    total_conversions,
    ROUND(total_clicks * 100.0 
        / NULLIF(total_impressions, 0), 
    2) AS ctr_percentage,
    
    ROUND(total_conversions * 100.0 
        / NULLIF(total_clicks, 0), 
    2) AS conversion_rate_percentage,
    
    ROUND(total_conversions * 100.0 
    / NULLIF(total_impressions, 0), 
    2) AS overall_funnel_conversion_percentage
FROM 
    funnel
ORDER BY 
    channel_name, overall_funnel_conversion_percentage DESC;
""", engine)

,channel_id,channel_name,campaign_id,campaign_name,total_impressions,total_clicks,total_conversions,ctr_percentage,conversion_rate_percentage,overall_funnel_conversion_percentage
0,CH-005,Email Marketing,CMP-004,Win-Back Users,31721,961,66,3.03,6.87,0.21
1,CH-002,Google Ads,CMP-002,Cashback Boost,40321,1205,92,2.99,7.63,0.23
2,CH-002,Google Ads,CMP-006,Trust & Security,15456,494,31,3.20,6.28,0.20
3,CH-001,Meta Ads,CMP-001,New Year Start Smart,47217,1406,108,2.98,7.68,0.23
4,CH-004,Referral Program,CMP-003,Refer & Earn,34416,1006,56,2.92,5.57,0.16
5,CH-003,Tiktok Ads,CMP-005,Card Activation,30869,928,67,3.01,7.22,0.22


*Funnel performance is fairly consistent across channels, with CTR values clustering around (3%), showing that ad engagement at the impression stage is broadly similar across all campaigns. However, differences become clearer at the conversion stage. Google Ads campaigns (especially “Cashback Boost”) and Meta Ads show the strongest overall funnel efficiency (0.23%), indicating better conversion effectiveness. TikTok Ads and Email Marketing perform moderately well, while Referral Program (“Refer & Earn”) has the weakest performance across all funnel metrics, especially conversion rate. At the campaign level, performance varies more within channels than between channels — for example, Google Ads has both a top-performing and a weaker campaign, showing that campaign strategy and targeting matter more than channel selection alone.*

## Project Recommendations
Based on the analysis of FinPay Digital’s marketing performance, several opportunities exist to improve customer acquisition efficiency, campaign profitability, and overall funnel performance.

* FinPay should prioritize increased investment in high-performing channels such as Meta Ads and selected Google Ads campaigns, as these channels demonstrated the strongest overall funnel conversion efficiency and signup performance.
* Campaign optimization efforts should focus more on post-click conversion performance rather than impression generation alone. While CTR remained relatively consistent across campaigns, conversion rates varied noticeably, indicating that landing experience, targeting quality, and offer relevance are the primary drivers of performance differences.
* The Referral Program requires strategic review despite recording high engagement rates. Its weaker conversion efficiency suggests that users may interact with referral campaigns without completing meaningful actions. Revising referral incentives, onboarding flow, or audience targeting may improve conversion outcomes.
* Email Marketing showed comparatively lower engagement efficiency, which may indicate audience fatigue or weaker content relevance. FinPay could improve performance by implementing better audience segmentation, personalization strategies, and optimized messaging frequency.
* Campaigns generating high impressions but low signup conversions should be reassessed to improve acquisition efficiency and reduce wasted marketing spend. This is particularly important for campaigns where reach significantly outweighs actual user acquisition outcomes.
* Since all campaigns produced negative estimated ROI when evaluated against conversion value, FinPay should improve budget allocation by prioritizing campaigns that generate higher-value transactional conversions rather than focusing primarily on acquisition volume.
* Funnel analysis revealed that the largest user drop-off occurs between impressions and clicks across all channels. This suggests a broader top-of-funnel engagement challenge, highlighting the need for stronger ad creatives, more targeted audience segmentation, and improved campaign messaging.
* Future analysis should incorporate Customer Acquisition Cost (CAC), customer retention metrics, and user lifetime value (LTV) to provide a more complete assessment of long-term campaign profitability and customer quality.

## Conclusion
This project analyzed FinPay Digital’s marketing performance to evaluate customer acquisition efficiency, campaign effectiveness, funnel conversion behaviour, and estimated ROI across multiple digital channels. Using SQL techniques such as CTEs, aggregations, joins, and KPI calculations, the analysis transformed raw marketing data into meaningful business insights.

One major challenge encountered was aligning business questions with the available dataset and interpreting metrics such as ROI using conversion value as a revenue proxy. The project also highlighted that high engagement or impressions do not always translate into strong conversion performance or profitability.

The analysis shows that campaign performance differences were driven more by conversion-stage efficiency than by initial ad engagement, with Meta Ads and selected Google Ads campaigns demonstrating stronger funnel performance compared to other channels.

A key lesson learned from this project is the importance of analysing the full customer journey rather than relying on a single marketing metric. Future improvements could include Customer Acquisition Cost (CAC), retention analysis, Customer Lifetime Value (LTV), and interactive dashboards using tools such as Power BI or Tableau to support deeper business decision-making.